In [1]:
import pandas as pd

load = pd.read_csv("../data/raw/load_germany_2023_2026.csv", sep=";")

load.head()

,Start date,End date,grid load [MWh] Calculated resolutions,Grid load incl. hydro pumped storage [MWh] Calculated resolutions,Hydro pumped storage [MWh] Calculated resolutions,Residual load [MWh] Calculated resolutions
0,"Jan 1, 2023 12:00 AM","Jan 1, 2023 1:00 AM","38,346.00","40,422.75","2,076.75","6,337.75"
1,"Jan 1, 2023 1:00 AM","Jan 1, 2023 2:00 AM","37,777.25","39,545.00","1,767.75","4,601.75"
2,"Jan 1, 2023 2:00 AM","Jan 1, 2023 3:00 AM","36,939.75","39,990.00","3,050.25","3,581.00"
3,"Jan 1, 2023 3:00 AM","Jan 1, 2023 4:00 AM","35,932.50","39,636.50","3,704.00","4,973.75"
4,"Jan 1, 2023 4:00 AM","Jan 1, 2023 5:00 AM","35,486.25","39,287.00","3,800.75","5,082.75"


In [2]:
load.columns

Index(['Start date', 'End date', 'grid load [MWh] Calculated resolutions',
       'Grid load incl. hydro pumped storage [MWh] Calculated resolutions',
       'Hydro pumped storage [MWh] Calculated resolutions',
       'Residual load [MWh] Calculated resolutions'],
      dtype='str')

In [3]:
load = load[[
    "Start date",
    "grid load [MWh] Calculated resolutions"
]]

In [4]:
load = load.rename(columns={
    "Start date": "timestamp",
    "grid load [MWh] Calculated resolutions": "load"
})

load.head()

,timestamp,load
0,"Jan 1, 2023 12:00 AM","38,346.00"
1,"Jan 1, 2023 1:00 AM","37,777.25"
2,"Jan 1, 2023 2:00 AM","36,939.75"
3,"Jan 1, 2023 3:00 AM","35,932.50"
4,"Jan 1, 2023 4:00 AM","35,486.25"


In [6]:
load["timestamp"] = pd.to_datetime(load["timestamp"])

load.head()


,timestamp,load
0,2023-01-01 00:00:00,"38,346.00"
1,2023-01-01 01:00:00,"37,777.25"
2,2023-01-01 02:00:00,"36,939.75"
3,2023-01-01 03:00:00,"35,932.50"
4,2023-01-01 04:00:00,"35,486.25"


In [7]:
load = load.sort_values("timestamp").reset_index(drop=True)


In [ ]:
full_range = pd.date_range(
    start=load["timestamp"].min(),
    end=load["timestamp"].max(),
    freq="h"
)

missing_timestamps = full_range.difference(load["timestamp"])
duplicate_count = load["timestamp"].duplicated().sum()

print("Rows:", len(load))
print("Expected rows:", len(full_range))
print("Missing timestamps:", len(missing_timestamps))
print("Duplicate timestamps:", duplicate_count)

print(missing_timestamps[:10])

Rows: 27264
Expected rows: 27264
Missing timestamps: 3
Duplicate timestamps: 3
DatetimeIndex(['2023-03-26 02:00:00', '2024-03-31 02:00:00',
               '2025-03-30 02:00:00'],
              dtype='datetime64[us]', freq=None)


In [16]:
load["load"] = load["load"].astype(str)
load["load"] = load["load"].str.replace(",", "", regex=False)
load["load"] = pd.to_numeric(load["load"], errors="coerce")
print(load.dtypes)


timestamp    datetime64[us]
load                float64
dtype: object


In [17]:
load = load.groupby("timestamp", as_index=False).mean()

In [20]:
print(load.head())
print(load.isnull().sum())
print(load["timestamp"].duplicated().sum())

            timestamp      load
0 2023-01-01 00:00:00  38346.00
1 2023-01-01 01:00:00  37777.25
2 2023-01-01 02:00:00  36939.75
3 2023-01-01 03:00:00  35932.50
4 2023-01-01 04:00:00  35486.25
timestamp    0
load         0
dtype: int64
0


In [60]:
print(load.shape)

load.columns

(27261, 2)


Index(['timestamp', 'load'], dtype='str')

In [45]:
import pandas as pd

gen = pd.read_csv("../data/raw/Actual_generation_2023_2026.csv", sep=";")

gen.head()

,Start date,End date,Biomass [MWh] Calculated resolutions,Hydropower [MWh] Calculated resolutions,Wind offshore [MWh] Calculated resolutions,Wind onshore [MWh] Calculated resolutions,Photovoltaics [MWh] Calculated resolutions,Other renewable [MWh] Calculated resolutions,Nuclear [MWh] Calculated resolutions,Lignite [MWh] Calculated resolutions,Hard coal [MWh] Calculated resolutions,Fossil gas [MWh] Calculated resolutions,Hydro pumped storage [MWh] Calculated resolutions,Other conventional [MWh] Calculated resolutions
0,"Jan 1, 2023 12:00 AM","Jan 1, 2023 1:00 AM","4,014.25","1,275.25","3,059.25","28,947.25",1.75,116.5,"2,459.50","3,859.25","2,067.50","1,593.75",125.25,"1,880.00"
1,"Jan 1, 2023 1:00 AM","Jan 1, 2023 2:00 AM","3,993.25","1,226.50","3,586.00","29,587.50",2.00,117.5,"2,458.75","3,866.50","2,052.00","1,437.00",302.50,"1,847.50"
2,"Jan 1, 2023 2:00 AM","Jan 1, 2023 3:00 AM","3,967.25","1,222.50","3,842.25","29,514.75",1.75,117.5,"2,459.75","3,860.25","2,034.25","1,435.00",141.50,"1,782.75"
3,"Jan 1, 2023 3:00 AM","Jan 1, 2023 4:00 AM","3,973.00","1,223.25","3,463.25","27,493.50",2.00,117.0,"2,460.50","3,864.75","2,037.00","1,432.50",96.25,"1,791.00"
4,"Jan 1, 2023 4:00 AM","Jan 1, 2023 5:00 AM","3,996.50","1,244.00","3,462.25","26,939.00",2.25,117.0,"2,461.00","3,841.00","2,040.25","1,430.75",188.25,"1,819.00"


In [46]:
gen = gen[[
    "Start date",
    "Wind offshore [MWh] Calculated resolutions",
    "Wind onshore [MWh] Calculated resolutions",
    "Photovoltaics [MWh] Calculated resolutions"
]]

In [47]:
gen.columns = [
    "timestamp",
    "wind_offshore",
    "wind_onshore",
    "solar"
]

gen.head()

,timestamp,wind_offshore,wind_onshore,solar
0,"Jan 1, 2023 12:00 AM","3,059.25","28,947.25",1.75
1,"Jan 1, 2023 1:00 AM","3,586.00","29,587.50",2.00
2,"Jan 1, 2023 2:00 AM","3,842.25","29,514.75",1.75
3,"Jan 1, 2023 3:00 AM","3,463.25","27,493.50",2.00
4,"Jan 1, 2023 4:00 AM","3,462.25","26,939.00",2.25


In [48]:
gen["wind_offshore"] = gen["wind_offshore"].astype(str)
gen["wind_offshore"] = gen["wind_offshore"].str.replace(",", "", regex=False)
gen["wind_offshore"] = pd.to_numeric(gen["wind_offshore"], errors="coerce")

gen["wind_onshore"] = gen["wind_onshore"].astype(str)
gen["wind_onshore"] = gen["wind_onshore"].str.replace(",", "", regex=False)
gen["wind_onshore"] = pd.to_numeric(gen["wind_onshore"], errors="coerce")

gen["wind"] = gen["wind_offshore"] + gen["wind_onshore"]

gen["solar"] = gen["solar"].astype(str)
gen["solar"] = gen["solar"].str.replace(",", "", regex=False)
gen["solar"] = pd.to_numeric(gen["solar"], errors="coerce")

gen.head()

,timestamp,wind_offshore,wind_onshore,solar,wind
0,"Jan 1, 2023 12:00 AM",3059.25,28947.25,1.75,32006.50
1,"Jan 1, 2023 1:00 AM",3586.00,29587.50,2.00,33173.50
2,"Jan 1, 2023 2:00 AM",3842.25,29514.75,1.75,33357.00
3,"Jan 1, 2023 3:00 AM",3463.25,27493.50,2.00,30956.75
4,"Jan 1, 2023 4:00 AM",3462.25,26939.00,2.25,30401.25


In [49]:
gen = gen[["timestamp", "wind", "solar"]]
gen.head()

,timestamp,wind,solar
0,"Jan 1, 2023 12:00 AM",32006.50,1.75
1,"Jan 1, 2023 1:00 AM",33173.50,2.00
2,"Jan 1, 2023 2:00 AM",33357.00,1.75
3,"Jan 1, 2023 3:00 AM",30956.75,2.00
4,"Jan 1, 2023 4:00 AM",30401.25,2.25


In [50]:
gen["timestamp"] = pd.to_datetime(gen["timestamp"])
gen = gen.sort_values("timestamp").reset_index(drop=True)

gen.head()

C:\Users\ankit\AppData\Local\Temp\ipykernel_14552\1281492872.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gen["timestamp"] = pd.to_datetime(gen["timestamp"])


,timestamp,wind,solar
0,2023-01-01 00:00:00,32006.50,1.75
1,2023-01-01 01:00:00,33173.50,2.00
2,2023-01-01 02:00:00,33357.00,1.75
3,2023-01-01 03:00:00,30956.75,2.00
4,2023-01-01 04:00:00,30401.25,2.25


In [51]:
gen.isnull().sum()

timestamp    0
wind         0
solar        0
dtype: int64

In [52]:
gen["timestamp"].duplicated().sum()

np.int64(3)

In [53]:
print(gen.dtypes)


timestamp    datetime64[us]
wind                float64
solar               float64
dtype: object


In [55]:
gen = gen.groupby("timestamp", as_index=False).mean()

In [56]:
gen["timestamp"].duplicated().sum()

np.int64(0)

In [61]:
print(gen.shape)

print (gen.columns)

gen.head()


(27261, 3)
Index(['timestamp', 'wind', 'solar'], dtype='str')


,timestamp,wind,solar
0,2023-01-01 00:00:00,32006.50,1.75
1,2023-01-01 01:00:00,33173.50,2.00
2,2023-01-01 02:00:00,33357.00,1.75
3,2023-01-01 03:00:00,30956.75,2.00
4,2023-01-01 04:00:00,30401.25,2.25


In [79]:
price = pd.read_csv("../data/processed/clean_price_data.csv", parse_dates=["timestamp"])
price.head()

,timestamp,price
0,2023-01-01 00:00:00,-5.17
1,2023-01-01 01:00:00,-1.07
2,2023-01-01 02:00:00,-1.47
3,2023-01-01 03:00:00,-5.08
4,2023-01-01 04:00:00,-4.49


In [80]:
print(price.columns)
print(load.columns)
print(gen.columns)

Index(['timestamp', 'price'], dtype='str')
Index(['timestamp', 'load'], dtype='str')
Index(['timestamp', 'wind', 'solar'], dtype='str')


In [81]:
df = price.merge(load, on="timestamp", how="inner")
df = df.merge(gen, on="timestamp", how="inner")

In [82]:
df = df.sort_values("timestamp")
df = df.set_index("timestamp")

In [83]:
df.head()


,price,load,wind,solar
timestamp,,,,
2023-01-01 00:00:00,-5.17,38346.00,32006.50,1.75
2023-01-01 01:00:00,-1.07,37777.25,33173.50,2.00
2023-01-01 02:00:00,-1.47,36939.75,33357.00,1.75
2023-01-01 03:00:00,-5.08,35932.50,30956.75,2.00
2023-01-01 04:00:00,-4.49,35486.25,30401.25,2.25


In [84]:
df.info()


<class 'pandas.DataFrame'>
DatetimeIndex: 27261 entries, 2023-01-01 00:00:00 to 2026-02-09 23:00:00
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   price   27261 non-null  float64
 1   load    27261 non-null  float64
 2   wind    27261 non-null  float64
 3   solar   27261 non-null  float64
dtypes: float64(4)
memory usage: 1.0 MB


In [85]:
df.isnull().sum()


price    0
load     0
wind     0
solar    0
dtype: int64

In [86]:
len(df)

27261

In [87]:
df.to_csv("../data/processed/final_dataset.csv")